# Treinamento com interface de alto nível

## Importação das bibliotecas

In [1]:
# http://pytorch.org/
from os.path import exists

import torch

In [2]:
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR

## Criação da rede

In [3]:
input_size = 32*32*3
input_size

3072

In [6]:
class CifarNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(input_size, 512)
    self.fc2 = nn.Linear(512, 256)
    self.fc3 = nn.Linear(256, 128)
    self.fc4 = nn.Linear(128, 64)
    self.fc5 = nn.Linear(64, 10)

  def forward(self, x):
    x = x.view(x.shape[0], -1)
    x = self.fc1(x)
    x = F.relu(x)
    x = self.fc2(x)
    x = F.relu(x)
    x = self.fc3(x)
    x = F.relu(x)
    x = self.fc4(x)
    x = F.relu(x)
    x = self.fc5(x)
    output = F.log_softmax(x, dim=1)
    return output

model = CifarNet()
model

CifarNet(
  (fc1): Linear(in_features=3072, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=64, bias=True)
  (fc5): Linear(in_features=64, out_features=10, bias=True)
)

In [7]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
    ])
dataset_train = datasets.CIFAR10('../data', train=True, download=True, transform=transform)
train_kwargs = {'batch_size': 64}


train_loader = torch.utils.data.DataLoader(dataset_train,**train_kwargs)

100%|██████████| 170498071/170498071 [00:12<00:00, 13148959.92it/s]


Extracting ../data/cifar-10-python.tar.gz to ../data


In [8]:
for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data, target
    print(data.shape)
    aux = data.view(data.shape[0], -1)
    print(aux.shape)

torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
torch.Size([64, 3072])
torch.Size([64, 3, 32, 32])
tor

KeyboardInterrupt: 

## Treinamento

### Criando o objeto de treinamento

In [9]:
def train(log_interval, dry_run, model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            if dry_run:
                break

In [10]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

## Avaliação

In [12]:
use_cuda = torch.cuda.is_available()

torch.manual_seed(1111)

device = torch.device("cuda" if use_cuda else "cpu")

train_kwargs = {'batch_size': 500}
test_kwargs = {'batch_size': 100}
if use_cuda:
    cuda_kwargs = {'num_workers': 1,
                    'pin_memory': True,
                    'shuffle': True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
    ])
dataset_train = datasets.CIFAR10('../data', train=True, download=True,
                    transform=transform)
dataset_test = datasets.CIFAR10('../data', train=False, download=True,
                    transform=transform)
train_loader = torch.utils.data.DataLoader(dataset_train,**train_kwargs)
test_loader = torch.utils.data.DataLoader(dataset_test, **test_kwargs)

model = CifarNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 14
scheduler = StepLR(optimizer, step_size=10, gamma=0.7)

for epoch in range(1, epochs + 1):
    train(100, False, model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)
    scheduler.step()

torch.save(model.state_dict(), "mnist_cnn.pt")

Files already downloaded and verified
Files already downloaded and verified
Train Epoch: 1 [0/50000 (0%)]	Loss: 2.305685

Test set: Average loss: 1.7470, Accuracy: 3770/10000 (38%)

Train Epoch: 2 [0/50000 (0%)]	Loss: 1.768868

Test set: Average loss: 1.6112, Accuracy: 4183/10000 (42%)

Train Epoch: 3 [0/50000 (0%)]	Loss: 1.496388

Test set: Average loss: 1.5044, Accuracy: 4674/10000 (47%)

Train Epoch: 4 [0/50000 (0%)]	Loss: 1.461850

Test set: Average loss: 1.5152, Accuracy: 4660/10000 (47%)

Train Epoch: 5 [0/50000 (0%)]	Loss: 1.433463

Test set: Average loss: 1.4584, Accuracy: 4847/10000 (48%)

Train Epoch: 6 [0/50000 (0%)]	Loss: 1.329879

Test set: Average loss: 1.4361, Accuracy: 4898/10000 (49%)

Train Epoch: 7 [0/50000 (0%)]	Loss: 1.285127

Test set: Average loss: 1.3707, Accuracy: 5153/10000 (52%)

Train Epoch: 8 [0/50000 (0%)]	Loss: 1.273093

Test set: Average loss: 1.3996, Accuracy: 5071/10000 (51%)

Train Epoch: 9 [0/50000 (0%)]	Loss: 1.364102

Test set: Average loss: 1.3797

In [ ]:
use_cuda = torch.cuda.is_available()

torch.manual_seed(1111)

device = torch.device("cuda" if use_cuda else "cpu")

train_kwargs = {'batch_size': 64}
test_kwargs = {'batch_size': 64}
if use_cuda:
    cuda_kwargs = {'num_workers': 1,
                    'pin_memory': True,
                    'shuffle': True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
    ])
dataset_train = datasets.FashionMNIST('../data', train=True, download=True,
                    transform=transform)
dataset_test = datasets.FashionMNIST('../data', train=False, download=True,
                    transform=transform)
train_loader = torch.utils.data.DataLoader(dataset_train,**train_kwargs)
test_loader = torch.utils.data.DataLoader(dataset_test, **test_kwargs)

modelFC = FCNet().to(device)
optimizer = optim.Adam(modelFC.parameters(), lr=0.001)

epochs = 14
scheduler = StepLR(optimizer, step_size=1, gamma=0.7)

for epoch in range(1, epochs + 1):
    train(100, False, modelFC, device, train_loader, optimizer, epoch)
    test(modelFC, device, test_loader)
    scheduler.step()

torch.save(modelFC.state_dict(), "mnist_cnn.pt")

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.308969
Train Epoch: 1 [6400/60000 (11%)]	Loss: 0.593600
Train Epoch: 1 [12800/60000 (21%)]	Loss: 0.253165
Train Epoch: 1 [19200/60000 (32%)]	Loss: 0.506283
Train Epoch: 1 [25600/60000 (43%)]	Loss: 0.570689
Train Epoch: 1 [32000/60000 (53%)]	Loss: 0.650853
Train Epoch: 1 [38400/60000 (64%)]	Loss: 0.368136
Train Epoch: 1 [44800/60000 (75%)]	Loss: 0.342323
Train Epoch: 1 [51200/60000 (85%)]	Loss: 0.392549
Train Epoch: 1 [57600/60000 (96%)]	Loss: 0.428142

Test set: Average loss: 0.4323, Accuracy: 8460/10000 (85%)

Train Epoch: 2 [0/60000 (0%)]	Loss: 0.516658
Train Epoch: 2 [6400/60000 (11%)]	Loss: 0.380985
Train Epoch: 2 [12800/60000 (21%)]	Loss: 0.319873
Train Epoch: 2 [19200/60000 (32%)]	Loss: 0.186250
Train Epoch: 2 [25600/60000 (43%)]	Loss: 0.303832
Train Epoch: 2 [32000/60000 (53%)]	Loss: 0.333673
Train Epoch: 2 [38400/60000 (64%)]	Loss: 0.155733
Train Epoch: 2 [44800/60000 (75%)]	Loss: 0.507623
Train Epoch: 2 [51200/60000 (85%)]	Loss: 0.247434
T